In [ ]:
import os, sys, json, time, copy, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

In [ ]:
try:
    from spikingjelly.activation_based import neuron, functional, surrogate
    from spikingjelly.activation_based.neuron import IzhikevichNode
    SJ_OK = True
    print("✅ SpikingJelly loaded")
except ImportError:
    SJ_OK = False
    print("⚠️  SpikingJelly not found — using built-in Izhikevich fallback")
    print("     Run: !pip install spikingjelly -q")


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} "
      f"({'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

In [ ]:
# GLOBAL CONFIGURATION
SEED        = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE    = 128      # resize images to 128×128 (good Colab speed/accuracy tradeoff)
BATCH_SIZE  = 32
T           = 8        # SNN temporal window (# of timesteps per forward pass)
SNN_DIM     = 256      # SNN hidden layer width
CNN_FEATS   = 512      # CNN backbone output features
NUM_CLASSES = 38       # PlantVillage: 38 disease / healthy classes
EPOCHS      = 20       # increase to 40+ for production runs
LR          = 1e-3
WEIGHT_DECAY= 1e-4
SUBSET_FRAC = 0.3      # use 30% of dataset — set 1.0 for full run
DATA_ROOT   = './PlantVillage-Dataset/raw/color'


In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\nConfig → IMG:{IMG_SIZE} | T:{T} | SNN_DIM:{SNN_DIM} | "
      f"EPOCHS:{EPOCHS} | SUBSET:{SUBSET_FRAC*100:.0f}%")
print(f"Device  → {DEVICE}\n")

In [ ]:
# IZHIKEVICH NEURON TYPE DEFINITIONS
NEURON_CONFIGS = {

    # ── RS : Regular Spiking ─────────────────────────────────────────────
    'RS': {
        'a': 0.02, 'b': 0.20, 'c': -65.0, 'd': 8.0,
        'color': '#4C72B0',
        'name':  'Regular Spiking',

        'brain_region': (
            'Cortical Pyramidal Neurons — Layer II/III & V '
            '(Primary Visual Cortex V1, Prefrontal Cortex)'
        ),
        'neuroscience': (
            'RS neurons are the most common excitatory cell type in the '
            'mammalian neocortex. They fire tonically with spike-frequency '
            'adaptation: an initial high rate that slows as the stimulus '
            'persists. Found in L2/3 and L5 pyramidal cells of V1, V2, and '
            'prefrontal areas. They are the principal substrate of sustained '
            'sensory representation and working memory.'
        ),
        'image_suitability': (
            'GOOD for sustained feature tracking across timesteps. '
            'Adaptation acts as a natural high-pass temporal filter, keeping '
            'the SNN from saturating on uniform texture regions. Analogous to '
            'simple-cell responses in V1 — ideal for encoding low-frequency '
            'spatial patterns like leaf shape and vein structure.'
        ),
        'expected_rank': 2,
    },

    # ── FS : Fast Spiking ─────────────────────────────────────────────────
    'FS': {
        'a': 0.10, 'b': 0.20, 'c': -65.0, 'd': 2.0,
        'color': '#DD8452',
        'name':  'Fast Spiking',

        'brain_region': (
            'Cortical GABAergic Interneurons — Parvalbumin+ Basket & '
            'Chandelier Cells (All cortical areas; dense in V1 Layer IV)'
        ),
        'neuroscience': (
            'FS neurons fire at very high rates (200–800 Hz) without adaptation. '
            'They are inhibitory interneurons (PV+) providing feed-forward and '
            'lateral inhibition that sharpens spatial selectivity and synchronises '
            'gamma oscillations (30–80 Hz). They receive strong thalamocortical '
            'input and gate information flow into the cortex.'
        ),
        'image_suitability': (
            'GOOD for rapid transient encoding — detects edges, contours, '
            'and abrupt colour boundaries with high temporal precision. '
            'High firing rate can overload the readout layer, leading to '
            'saturation; benefits from careful weight initialisation. '
            'Analogous to end-stopped cells and motion-sensitive cells in MT.'
        ),
        'expected_rank': 4,
    },

    # ── CH : Chattering ───────────────────────────────────────────────────
    'CH': {
        'a': 0.02, 'b': 0.20, 'c': -50.0, 'd': 2.0,
        'color': '#55A868',
        'name':  'Chattering',

        'brain_region': (
            'Visual Cortex Layer II/III Chattering Cells '
            '(Cat & Primate V1/V2 — Gray & McCormick 1996)'
        ),
        'neuroscience': (
            'Chattering (CH) neurons fire high-frequency bursts (300–700 Hz '
            'intra-burst) separated by quiescent intervals, producing a '
            '"chattering" pattern. First discovered in cat visual cortex by '
            'Gray & McCormick (1996). They drive gamma-band synchrony critical '
            'for figure-ground segregation and perceptual binding. c=−50 means '
            'the reset voltage is significantly depolarised, enabling rapid '
            'burst re-initiation.'
        ),
        'image_suitability': (
            '⭐ OPTIMAL for image classification. Burst-then-silence naturally '
            'encodes complex multi-scale visual features: texture statistics, '
            'contour curvature, colour gradients. The burst-silence temporal '
            'structure acts as a built-in attention gate separating foreground '
            '(disease lesions) from background (healthy leaf). Directly mirrors '
            'the biological mechanism of scene understanding in primate V1/V2.'
        ),
        'expected_rank': 1,
    },

    # ── IB : Intrinsic Bursting ───────────────────────────────────────────
    'IB': {
        'a': 0.02, 'b': 0.20, 'c': -55.0, 'd': 4.0,
        'color': '#C44E52',
        'name':  'Intrinsic Bursting',

        'brain_region': (
            'Neocortical Layer V Thick-Tufted Pyramidal Cells '
            '(Primary Motor Cortex, Somatosensory Cortex, V1 Layer Vb)'
        ),
        'neuroscience': (
            'IB neurons fire an initial burst of 2–5 spikes followed by '
            'repetitive single spikes. They are large Layer-V pyramidal cells '
            'with thick apical dendrites projecting to subcortical structures '
            '(thalamus, brainstem, spinal cord). The initial burst acts as a '
            '"salience alarm" — signalling the onset of a new stimulus — while '
            'the sustained tail encodes stimulus persistence.'
        ),
        'image_suitability': (
            'STRONG for detecting salient, localised features. The burst onset '
            'fires strongly to novel visual events (disease spots appearing on '
            'leaf surface), making IB excellent for detecting localised pathology '
            'against a background. Weaker at encoding uniformly distributed '
            'symptoms like diffuse yellowing. Competes well with CH overall.'
        ),
        'expected_rank': 3,
    },

    # ── LTS : Low Threshold Spiking ───────────────────────────────────────
    'LTS': {
        'a': 0.02, 'b': 0.25, 'c': -65.0, 'd': 2.0,
        'color': '#8172B2',
        'name':  'Low Threshold Spiking',

        'brain_region': (
            'Cortical Somatostatin+ Martinotti Interneurons '
            '(Layers I–IV; dense in barrel cortex & V1 supragranular layers)'
        ),
        'neuroscience': (
            'LTS (also called low-threshold calcium-spike) neurons have a '
            'depolarised b=0.25 that produces a low activation threshold and '
            'burst on rebound after hyperpolarisation (post-inhibitory rebound '
            'burst). Found in SST+ Martinotti interneurons that inhibit '
            'dendritic compartments of pyramidal cells. They modulate local '
            'dendritic computation and provide layer-selective inhibition.'
        ),
        'image_suitability': (
            'GOOD for detecting subtle, low-contrast disease symptoms. '
            'The low firing threshold means responses to weak CNN feature inputs, '
            'making LTS sensitive to early-stage disease where discolouration '
            'is faint. However, the rebound-burst mechanism can produce spurious '
            'activations from noise, potentially increasing false-positive rate.'
        ),
        'expected_rank': 5,
    },
}


In [ ]:
NEURON_KEYS = list(NEURON_CONFIGS.keys())
print("Neuron types configured →", NEURON_KEYS)

In [ ]:
def get_transforms(train: bool = True):
    """ImageNet-normalised transforms with augmentation for training."""
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if train:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.2, hue=0.05),
            transforms.RandomGrayscale(p=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

In [ ]:
class TransformDataset(torch.utils.data.Dataset):
    """Wrapper to apply a different transform to a Subset."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img is already a tensor if parent had transforms; reload from PIL
        # To handle this cleanly we work with a fresh ImageFolder per split
        return img, label

In [ ]:
def load_plantvillage(data_root: str = DATA_ROOT,
                      subset_frac: float = SUBSET_FRAC):
    """
    Load PlantVillage from an ImageFolder-compatible directory.
    Returns (loaders dict, num_classes, split sizes dict).
    """
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(
            f"\n❌ Dataset not found at: {data_root}\n\n"
            "Download options:\n"
            "  Option A (GitHub):\n"
            "    !git clone --depth 1 "
            "https://github.com/spMohanty/PlantVillage-Dataset.git\n"
            "    DATA_ROOT = './PlantVillage-Dataset/raw/color'\n\n"
            "  Option B (Kaggle):\n"
            "    !pip install kaggle\n"
            "    !kaggle datasets download -d emmarex/plantdisease "
            "-p ./data --unzip\n"
            "    DATA_ROOT = './data/PlantVillage'\n"
        )

    # Build three separate ImageFolder datasets with correct transforms
    ds_train = ImageFolder(data_root, transform=get_transforms(train=True))
    ds_eval  = ImageFolder(data_root, transform=get_transforms(train=False))

    num_classes = len(ds_train.classes)
    n_total     = len(ds_train)

    print(f"✅ PlantVillage loaded — {n_total:,} images | {num_classes} classes")
    print(f"   First 5: {ds_train.classes[:5]}")
    print(f"   Last  3: {ds_train.classes[-3:]}")

    # Optional subset (for fast Colab runs)
    if subset_frac < 1.0:
        n_sub   = int(n_total * subset_frac)
        indices = random.sample(range(n_total), n_sub)
        print(f"   Subset  : {subset_frac*100:.0f}% → {n_sub:,} samples")
    else:
        indices = list(range(n_total))

    n      = len(indices)
    n_tr   = int(0.70 * n)
    n_vl   = int(0.15 * n)
    n_te   = n - n_tr - n_vl

    rng    = torch.Generator().manual_seed(SEED)
    perm   = torch.randperm(n, generator=rng).tolist()
    idx_tr = [indices[i] for i in perm[:n_tr]]
    idx_vl = [indices[i] for i in perm[n_tr:n_tr+n_vl]]
    idx_te = [indices[i] for i in perm[n_tr+n_vl:]]

    tr_ds = Subset(ds_train, idx_tr)
    vl_ds = Subset(ds_eval,  idx_vl)
    te_ds = Subset(ds_eval,  idx_te)

    mk_loader = lambda ds, shuf: DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuf,
        num_workers=2, pin_memory=torch.cuda.is_available(), drop_last=False
    )
    loaders = {'train': mk_loader(tr_ds, True),
               'val':   mk_loader(vl_ds, False),
               'test':  mk_loader(te_ds, False)}
    splits  = {'train': n_tr, 'val': n_vl, 'test': n_te}
    print(f"   Splits  : train={n_tr:,} | val={n_vl:,} | test={n_te:,}\n")
    return loaders, num_classes, splits

In [ ]:
# CNN feature extractor

class CNNBackbone(nn.Module):
    """
    4-block convolutional feature extractor.
    Input  : [B, 3, IMG_SIZE, IMG_SIZE]
    Output : [B, CNN_FEATS]     continuous feature vector (node F output)

    Block structure mirrors diagram nodes C→D→E→F:
      [Conv→BN→ReLU→Pool] × 3  +  AdaptiveAvgPool  +  Dense

    Activations: ReLU (continuous) — no spikes here.
    """
    def __init__(self, out_dim: int = CNN_FEATS):
        super().__init__()

        def conv_block(in_ch, out_ch, pool=True):
            layers = [
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.MaxPool2d(2, 2))   # halves spatial dims
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(3,   32),    # 128→64  (Conv Layer 1, node C)
            conv_block(32,  64),    # 64→32   (Pooling Layer, node D)
            conv_block(64,  128),   # 32→16   (Conv Layer N, node E)
            conv_block(128, 256, pool=False),  # feature depth
            nn.AdaptiveAvgPool2d((4, 4)),       # spatial → 4×4 fixed (node D→F)
        )

        # Flattening + Dense layer (node F)
        self.dense = nn.Sequential(
            nn.Flatten(),                           # 256×4×4 = 4096
            nn.Linear(256 * 4 * 4, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dense(self.features(x))  # [B, out_dim]


In [ ]:
# Izhikevich node
class IzhikevichCell(nn.Module):
    """
    Differentiable Izhikevich neuron layer.

    Uses a Surrogate Gradient (ATan) for ∂spike/∂v so backpropagation
    flows through the discrete spike event during training.

    If SpikingJelly is available its native IzhikevichNode is used;
    otherwise this pure-PyTorch fallback is identical numerically.

    State variables (per neuron per batch item):
      v : membrane potential  (mV)    — initialised to c (reset value)
      u : recovery variable           — initialised to b*c
    """
    def __init__(self, a: float, b: float, c: float, d: float,
                 v_thresh: float = 30.0, dt: float = 1.0):
        super().__init__()
        self.a, self.b, self.c_val, self.d = a, b, c, d
        self.v_thresh = v_thresh
        self.dt       = dt
        # State (initialised lazily)
        self.v: torch.Tensor = None
        self.u: torch.Tensor = None

    def _reset_state(self, batch: int, n: int, dev: torch.device):
        self.v = torch.full((batch, n), self.c_val, device=dev)
        self.u = torch.full((batch, n), self.b * self.c_val, device=dev)

    def reset(self):
        """Call between independent samples / sequences."""
        self.v = None
        self.u = None

    def forward(self, I: torch.Tensor) -> torch.Tensor:
        """
        I : [B, N]  input current (output of a Linear layer)
        Returns spike tensor [B, N] ∈ {0, 1}
        """
        B, N = I.shape
        dev  = I.device
        if self.v is None or self.v.shape != (B, N):
            self._reset_state(B, N, dev)

        v = self.v.detach(); u = self.u.detach()

        # Izhikevich dynamics (Euler, dt=1 ms)
        dv = (0.04 * v * v + 5.0 * v + 140.0 - u + I) * self.dt
        du = (self.a * (self.b * v - u)) * self.dt
        v  = v + dv
        u  = u + du

        # ── Surrogate-gradient spike (ATan approximation) ──
        # Forward  : spike = 1 if v >= thresh, else 0
        # Backward : ∂spike/∂v ≈ 1 / (π/2 · (1 + (πv/2)²))  [ATan surrogate]
        spike = _SurrogateATan.apply(v - self.v_thresh)

        # Reset rule after spike
        v_reset = torch.where(spike.bool(), torch.full_like(v, self.c_val), v)
        u_after = u + self.d * spike

        self.v = v_reset
        self.u = u_after
        return spike                          # [B, N]

In [ ]:
class _SurrogateATan(torch.autograd.Function):
    """
    ATan surrogate gradient for spike generation.
    Forward  : Heaviside(x)  — spike if x ≥ 0
    Backward : d/dx [arctan(x·π/2) / π + 0.5]  (smooth proxy)
    """
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x >= 0.).float()

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        alpha = 2.0
        grad  = alpha / (2. * (1. + (alpha * x).pow(2))) * grad_output
        return grad

In [ ]:
# SNN layer wrapper
class SNNLayer(nn.Module):
    """
    One linear-connected Izhikevich SNN layer.
    Accepts time-major input [T, B, in_dim],
    returns spike sequence    [T, B, out_dim].

    Optionally uses SpikingJelly's IzhikevichNode for performance.
    """
    def __init__(self, in_dim: int, out_dim: int, neuron_type: str):
        super().__init__()
        cfg = NEURON_CONFIGS[neuron_type]
        self.fc = nn.Linear(in_dim, out_dim, bias=True)
        nn.init.xavier_normal_(self.fc.weight)

        if SJ_OK:
            # SpikingJelly's native node (CUDA-optimised)
            self.izh = IzhikevichNode(
                a=cfg['a'], b=cfg['b'], c=cfg['c'], d=cfg['d'],
                v_threshold=30.0,
                surrogate_function=surrogate.ATan(),
                step_mode='s',
            )
            self._use_sj = True
        else:
            self.izh = IzhikevichCell(
                a=cfg['a'], b=cfg['b'], c=cfg['c'], d=cfg['d'],
                v_thresh=30.0,
            )
            self._use_sj = False

    def reset(self):
        if self._use_sj:
            functional.reset_net(self.izh)
        else:
            self.izh.reset()

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        """x_seq: [T, B, in_dim]  →  spikes: [T, B, out_dim]"""
        T_steps  = x_seq.shape[0]
        out_list = []
        for t in range(T_steps):
            curr  = self.fc(x_seq[t])          # [B, out_dim]  linear transform
            spike = self.izh(curr)             # [B, out_dim]  Izhikevich fire
            out_list.append(spike)
        return torch.stack(out_list, dim=0)    # [T, B, out_dim]



In [ ]:
# STDP learning module
class STDPUpdater:
    """
    Trace-based Spike-Timing Dependent Plasticity.

    Applied as an auxiliary Hebbian update to fc weights of an SNNLayer
    AFTER the backprop gradient step (combined learning rule).

    Pre-synaptic trace  : x_pre  ← x_pre·exp(−1/τ_pre)  + s_pre
    Post-synaptic trace : x_post ← x_post·exp(−1/τ_post) + s_post
    Weight change:
      ΔW += A+ · x_pre^T · s_post    (potentiation: post fires AFTER pre)
      ΔW −= A− · s_pre^T · x_post    (depression:   pre fires AFTER post)
    """
    def __init__(self, tau_pre: float = 20., tau_post: float = 20.,
                 A_plus: float = 0.008, A_minus: float = 0.010,
                 lr_stdp: float = 5e-4, clip: float = 0.005):
        self.tau_pre  = tau_pre
        self.tau_post = tau_post
        self.A_plus   = A_plus
        self.A_minus  = A_minus
        self.lr       = lr_stdp
        self.clip     = clip
        self._x_pre   = None
        self._x_post  = None

    def reset(self):
        self._x_pre  = None
        self._x_post = None

    @torch.no_grad()
    def step(self, layer: SNNLayer,
             pre_spikes:  torch.Tensor,   # [T, B, in_dim]
             post_spikes: torch.Tensor):  # [T, B, out_dim]
        T_steps, B, in_dim  = pre_spikes.shape
        _, _, out_dim        = post_spikes.shape
        dev = pre_spikes.device

        if self._x_pre is None:
            self._x_pre  = torch.zeros(B, in_dim,  device=dev)
            self._x_post = torch.zeros(B, out_dim, device=dev)

        dW = torch.zeros_like(layer.fc.weight)  # [out_dim, in_dim]
        ep = torch.tensor(-1.0)

        for t in range(T_steps):
            s_pre  = pre_spikes[t]    # [B, in_dim]
            s_post = post_spikes[t]   # [B, out_dim]

            # Update eligibility traces
            decay_pre  = math.exp(-1.0 / self.tau_pre)
            decay_post = math.exp(-1.0 / self.tau_post)
            self._x_pre  = self._x_pre  * decay_pre  + s_pre
            self._x_post = self._x_post * decay_post + s_post

            # Potentiation: post fires → strengthen pre→post connection
            dW += self.A_plus  * torch.einsum('bo,bi->oi', s_post,        self._x_pre)
            # Depression:   pre fires → weaken if post hasn't fired
            dW -= self.A_minus * torch.einsum('bo,bi->oi', self._x_post,  s_pre)

        dW /= B  # average over batch
        layer.fc.weight.data.add_(self.lr * dW.clamp(-self.clip, self.clip))



In [ ]:
# Full hybrid model
class HybridCNNSNN(nn.Module):
    """
    Complete Hybrid CNN-SNN architecture.

    Diagram mapping:
      A  → raw image input
      B–F → self.cnn  (CNNBackbone)
      G   → self.encoder  (rate-coding spike generator)
      H   → spike train tensor [T, B, SNN_DIM]
      I   → self.snn_hidden  (SNNLayer with neuron-type-specific IzhikevichCell)
      J/K/L/M/N → the Izhikevich neurons inside snn_hidden (one per model)
      O   → self.snn_integrate  (second SNN layer for temporal integration)
      P   → self.readout
      Q   → softmax
      R   → argmax predicted class
    """
    def __init__(self, neuron_type: str, num_classes: int,
                 cnn_feats: int = CNN_FEATS,
                 snn_dim:   int = SNN_DIM,
                 T:         int = T):
        super().__init__()
        self.neuron_type = neuron_type
        self.T           = T
        self.num_classes = num_classes

        # ── CNN Phase ───────────────────────────────────────── nodes B→F
        self.cnn = CNNBackbone(out_dim=cnn_feats)

        # ── Spike Encoder (Rate Coding) ────────────────────── nodes G→H
        # Maps continuous CNN features → firing-probability per neuron
        self.encoder = nn.Sequential(
            nn.Linear(cnn_feats, snn_dim),
            nn.Sigmoid(),     # output ∈ (0,1) → Bernoulli firing probability
        )

        # ── SNN Hidden Layer ─────────────────────────────────── node I
        # Contains J/K/L/M/N Izhikevich neurons depending on neuron_type
        self.snn_hidden = SNNLayer(snn_dim, snn_dim, neuron_type)

        # ── SNN Integration Layer ─────────────────────────────── node O
        # Temporal integration + dimensionality compression
        self.snn_integrate = SNNLayer(snn_dim, snn_dim // 2, neuron_type)

        # ── Readout + Classifier ──────────────────────────── nodes P→Q→R
        self.readout = nn.Sequential(
            nn.LayerNorm(snn_dim // 2),
            nn.Linear(snn_dim // 2, num_classes),
        )

        # Internal spike storage for STDP and analysis
        self._spk_encoder  : torch.Tensor = None
        self._spk_hidden   : torch.Tensor = None
        self._spk_integrate: torch.Tensor = None
        self._firing_stats : dict         = {}

    def _rate_encode(self, features: torch.Tensor) -> torch.Tensor:
        """
        Bernoulli rate coding: each CNN feature dimension → firing probability.
        features : [B, cnn_feats]
        returns  : [T, B, snn_dim]  binary spike trains
        """
        prob = self.encoder(features)                      # [B, snn_dim]
        # Expand along time axis and sample from Bernoulli
        prob_T = prob.unsqueeze(0).expand(self.T, -1, -1)  # [T, B, snn_dim]
        spikes = torch.bernoulli(prob_T)
        return spikes

    def reset_snn(self):
        """Reset all stateful SNN layers between independent samples."""
        self.snn_hidden.reset()
        self.snn_integrate.reset()

    def forward(self, x: torch.Tensor):
        """
        x : [B, 3, H, W]
        returns: (logits [B, C], firing_rate_dict)
        """
        # ① CNN feature extraction — continuous valued (nodes C→F)
        feats = self.cnn(x)                             # [B, CNN_FEATS]

        # ② Spike encoding — rate coding (nodes G→H)
        self.reset_snn()
        spk_input = self._rate_encode(feats)            # [T, B, SNN_DIM]
        self._spk_encoder = spk_input.detach()

        # ③ Izhikevich SNN hidden layer (node I — one of J/K/L/M/N)
        spk_hidden = self.snn_hidden(spk_input)         # [T, B, SNN_DIM]
        self._spk_hidden = spk_hidden.detach()

        # ④ Temporal integration (node O)
        spk_integ = self.snn_integrate(spk_hidden)      # [T, B, SNN_DIM//2]
        self._spk_integrate = spk_integ.detach()

        # ⑤ Mean-rate readout — average spike count over T (nodes P→Q)
        activity  = spk_integ.mean(dim=0)               # [B, SNN_DIM//2]
        logits    = self.readout(activity)              # [B, num_classes]

        # Record firing statistics
        self._firing_stats = {
            'encoder_fr':   spk_input.mean().item(),
            'hidden_fr':    spk_hidden.mean().item(),
            'integrate_fr': spk_integ.mean().item(),
        }
        return logits



In [ ]:
def make_model(nt: str, nc: int) -> HybridCNNSNN:
    m = HybridCNNSNN(nt, nc).to(DEVICE)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  [{nt}] {NEURON_CONFIGS[nt]['name']:22s} — {n:,} trainable params")
    return m

In [ ]:
# Trauining engine

def _one_epoch(model: HybridCNNSNN,
               loader: DataLoader,
               optimizer: optim.Optimizer,
               criterion: nn.Module,
               stdp: STDPUpdater,
               epoch: int) -> tuple:
    """
    Train one epoch.
    Returns (avg_loss, accuracy).
    """
    model.train()
    tot_loss = tot_correct = tot_n = 0

    bar = tqdm(loader,
               desc=f"  [{model.neuron_type}] Ep{epoch+1:02d}",
               leave=False, ncols=100)

    for imgs, labels in bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping prevents SNN weight explosion
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # ── STDP auxiliary update ─────────────────────────────────────
        # Apply Hebbian STDP on the fc weights of the hidden SNN layer
        # using the cached pre/post spikes from this forward pass.
        if (model._spk_encoder is not None and
                model._spk_hidden is not None):
            stdp.step(model.snn_hidden,
                      model._spk_encoder,
                      model._spk_hidden)
        stdp.reset()

        tot_loss    += loss.item() * imgs.size(0)
        preds        = logits.argmax(1)
        tot_correct += (preds == labels).sum().item()
        tot_n       += imgs.size(0)

        bar.set_postfix(loss=f"{loss.item():.4f}",
                        acc=f"{100.*tot_correct/tot_n:.1f}%")

    return tot_loss / tot_n, tot_correct / tot_n


In [ ]:
@torch.no_grad()
def _evaluate(model: HybridCNNSNN,
              loader: DataLoader,
              criterion: nn.Module) -> tuple:
    """Evaluate; return (avg_loss, accuracy, mean_firing_stats)."""
    model.eval()
    tot_loss = tot_correct = tot_n = 0
    fr_accum = defaultdict(float)
    n_batches = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        tot_loss    += loss.item() * imgs.size(0)
        preds        = logits.argmax(1)
        tot_correct += (preds == labels).sum().item()
        tot_n       += imgs.size(0)
        for k, v in model._firing_stats.items():
            fr_accum[k] += v
        n_batches += 1

    avg_fr = {k: v / max(n_batches, 1) for k, v in fr_accum.items()}
    return tot_loss / tot_n, tot_correct / tot_n, avg_fr

In [ ]:
def train_one_model(neuron_type: str,
                    loaders: dict,
                    num_classes: int,
                    epochs: int = EPOCHS) -> dict:
    """
    Full training + validation + test pipeline for one neuron type.
    Returns history dict.
    """
    cfg = NEURON_CONFIGS[neuron_type]
    print(f"\n{'═'*65}")
    print(f"  ▶ Training  [{neuron_type}]  {cfg['name']}")
    print(f"    Brain     : {cfg['brain_region']}")
    print(f"{'═'*65}")

    model     = make_model(neuron_type, num_classes)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    stdp      = STDPUpdater(lr_stdp=5e-4)

    hist = dict(
        neuron_type=neuron_type,
        train_loss=[], val_loss=[],
        train_acc=[],  val_acc=[],
        lr=[], firing_rates=[],
        best_val_acc=0., best_epoch=0,
    )

    best_wts   = None
    t_start    = time.time()

    for ep in range(epochs):
        tr_loss, tr_acc = _one_epoch(model, loaders['train'],
                                      optimizer, criterion, stdp, ep)
        vl_loss, vl_acc, vl_fr = _evaluate(model, loaders['val'], criterion)
        scheduler.step()

        hist['train_loss'].append(tr_loss)
        hist['val_loss'].append(vl_loss)
        hist['train_acc'].append(tr_acc * 100.)
        hist['val_acc'].append(vl_acc * 100.)
        hist['lr'].append(optimizer.param_groups[0]['lr'])
        hist['firing_rates'].append(dict(vl_fr))

        if vl_acc > hist['best_val_acc']:
            hist['best_val_acc'] = vl_acc
            hist['best_epoch']   = ep
            best_wts             = copy.deepcopy(model.state_dict())

        if (ep + 1) % 5 == 0 or ep == 0:
            print(f"    Ep {ep+1:02d}/{epochs} │ "
                  f"tr {tr_loss:.4f}/{tr_acc*100:.1f}% │ "
                  f"vl {vl_loss:.4f}/{vl_acc*100:.1f}% │ "
                  f"FR_hid={vl_fr.get('hidden_fr',0):.3f} │ "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    # ── Final test evaluation ──────────────────────────────────────────
    model.load_state_dict(best_wts)
    te_loss, te_acc, te_fr = _evaluate(model, loaders['test'], criterion)
    hist['test_loss']  = te_loss
    hist['test_acc']   = te_acc * 100.
    hist['test_fr']    = dict(te_fr)
    hist['train_time'] = time.time() - t_start

    print(f"\n    ✅  [{neuron_type}] best_val={hist['best_val_acc']*100:.2f}% "
          f"(ep {hist['best_epoch']+1}) │ "
          f"test={te_acc*100:.2f}% │ "
          f"time={hist['train_time']/60:.1f}min")

    # Save checkpoint
    ckpt = f"ckpt_{neuron_type}.pt"
    torch.save({'state_dict': best_wts, 'history': hist}, ckpt)
    print(f"    💾  Checkpoint → {ckpt}")

    del model
    torch.cuda.empty_cache()
    return hist

In [ ]:
def train_all(loaders: dict, num_classes: int, epochs: int = EPOCHS) -> dict:
    """
    Train all 5 neuron-type models sequentially (safe for Colab 16 GB VRAM).
    Returns dict[neuron_type → history].
    """
    all_hist = {}
    for nt in NEURON_KEYS:
        all_hist[nt] = train_one_model(nt, loaders, num_classes, epochs)

    # Summary table
    rows = []
    for nt in NEURON_KEYS:
        h = all_hist[nt]
        rows.append({
            'Type'          : nt,
            'Name'          : NEURON_CONFIGS[nt]['name'],
            'Best Val Acc%' : f"{h['best_val_acc']*100:.2f}",
            'Test Acc%'     : f"{h['test_acc']:.2f}",
            'Best Epoch'    : h['best_epoch'] + 1,
            'Time (min)'    : f"{h['train_time']/60:.1f}",
        })
    print("\n\n" + "═"*75)
    print("  TRAINING SUMMARY — All 5 Izhikevich Neuron Types")
    print("═"*75)
    print(pd.DataFrame(rows).to_string(index=False))
    print("═"*75)
    return all_hist

In [ ]:
# SPIKE PATTERN SIMULATOR
def simulate_izhikevich(neuron_type: str,
                        I_app: float = 10.0,
                        T_ms: int = 300) -> tuple:
    """
    Simulate bare Izhikevich neuron dynamics for visualisation.
    Returns (voltage trace, spike times).
    """
    cfg      = NEURON_CONFIGS[neuron_type]
    a, b, c, d = cfg['a'], cfg['b'], cfg['c'], cfg['d']
    v, u     = float(c), float(b * c)
    vs, ts   = [], []

    for t in range(T_ms):
        # Step current onset after 50 ms
        I = I_app if t >= 50 else 0.0
        dv = 0.04 * v * v + 5.0 * v + 140.0 - u + I
        du = a * (b * v - u)
        v += dv; u += du
        if v >= 30.:
            ts.append(t)
            v = float(c); u += d
        vs.append(min(v, 35.0))

    return vs, ts

In [ ]:
# Visualisations
def plot_per_model_curves(all_hist: dict):
    """5×2 grid: loss + accuracy curves for each neuron type."""
    fig, axes = plt.subplots(5, 2, figsize=(16, 22))
    fig.suptitle(
        'Hybrid CNN-SNN — Per-Model Training Curves\n'
        'Izhikevich Neuron Types · PlantVillage Dataset',
        fontsize=15, fontweight='bold', y=1.005
    )

    ep = list(range(1, EPOCHS + 1))
    for row, nt in enumerate(NEURON_KEYS):
        h   = all_hist[nt]
        cfg = NEURON_CONFIGS[nt]
        c   = cfg['color']

        # ── Loss ──────────────────────────────────────────────────────
        ax = axes[row, 0]
        ax.plot(ep, h['train_loss'], color=c, lw=2.0, label='Train')
        ax.plot(ep, h['val_loss'],   color=c, lw=2.0, ls='--', alpha=0.75, label='Val')
        ax.axvline(h['best_epoch']+1, color='#333', lw=1.2, ls=':', alpha=0.6,
                   label=f"Best ep {h['best_epoch']+1}")
        ax.fill_between(ep, h['train_loss'], h['val_loss'],
                        alpha=0.06, color=c)
        ax.set_title(f"[{nt}] {cfg['name']}  —  Loss", fontsize=10, fontweight='bold', color=c)
        ax.set_xlabel('Epoch', fontsize=9); ax.set_ylabel('Cross-Entropy Loss', fontsize=9)
        ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.25)

        # ── Accuracy ───────────────────────────────────────────────────
        ax = axes[row, 1]
        ax.plot(ep, h['train_acc'], color=c, lw=2.0, label='Train')
        ax.plot(ep, h['val_acc'],   color=c, lw=2.0, ls='--', alpha=0.75, label='Val')
        ax.axhline(h['test_acc'], color='#111', lw=1.5, ls='-.',
                   label=f"Test {h['test_acc']:.1f}%")
        ax.axvline(h['best_epoch']+1, color='#333', lw=1.2, ls=':', alpha=0.6)
        ax.fill_between(ep, h['train_acc'], h['val_acc'], alpha=0.06, color=c)
        ax.set_title(f"[{nt}] {cfg['name']}  —  Accuracy", fontsize=10, fontweight='bold', color=c)
        ax.set_xlabel('Epoch', fontsize=9); ax.set_ylabel('Accuracy (%)', fontsize=9)
        ax.set_ylim(0, 105)
        ax.legend(fontsize=8, loc='lower right'); ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig('01_per_model_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅  Saved: 01_per_model_curves.png")

In [ ]:
def plot_comparison_overlay(all_hist: dict):
    """Overlay all 5 models on shared val-loss / val-acc axes."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('CNN-SNN Model Comparison — Validation Metrics\n5 Izhikevich Neuron Types',
                 fontsize=13, fontweight='bold')

    ep = list(range(1, EPOCHS + 1))
    for nt, h in all_hist.items():
        c   = NEURON_CONFIGS[nt]['color']
        lbl = f"{nt}  {NEURON_CONFIGS[nt]['name']}"
        a1.plot(ep, h['val_loss'], color=c, lw=2.2, label=lbl)
        a2.plot(ep, h['val_acc'],  color=c, lw=2.2, label=lbl)

    for ax, ylabel, title in [
        (a1, 'Val Loss (CE)',    'Validation Loss'),
        (a2, 'Val Accuracy (%)', 'Validation Accuracy'),
    ]:
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(fontsize=9, loc='best')
        ax.grid(True, alpha=0.25)

    a2.set_ylim(0, 105)
    plt.tight_layout()
    plt.savefig('02_comparison_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅  Saved: 02_comparison_overlay.png")

In [ ]:
def plot_bar_and_firing(all_hist: dict):
    """Bar chart of test accuracy + grouped firing-rate bars."""
    nt_list    = list(all_hist.keys())
    test_accs  = [all_hist[nt]['test_acc']  for nt in nt_list]
    colors     = [NEURON_CONFIGS[nt]['color'] for nt in nt_list]
    xlabels    = [f"{nt}\n{NEURON_CONFIGS[nt]['name']}" for nt in nt_list]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Test Performance & Spiking Statistics', fontsize=13, fontweight='bold')
    # ── Test accuracy bar ──────────────────────────────────────────────
    bars = ax1.bar(xlabels, test_accs, color=colors, width=0.55,
                   edgecolor='white', linewidth=1.5)
    for bar, acc in zip(bars, test_accs):
        ax1.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.4, f"{acc:.1f}%",
                 ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax1.set_title('Test Accuracy by Neuron Type', fontweight='bold')
    ax1.set_ylabel('Test Accuracy (%)', fontsize=10)
    ax1.set_ylim(0, 110); ax1.grid(True, alpha=0.25, axis='y')

    # ── Firing-rate grouped bar ────────────────────────────────────────
    fr_keys = ['encoder_fr', 'hidden_fr', 'integrate_fr']
    fr_lbls = ['Spike\nEncoder', 'SNN\nHidden', 'Integration\nLayer']
    x       = np.arange(len(fr_keys))
    w       = 0.15
    for i, nt in enumerate(nt_list):
        fr   = all_hist[nt].get('test_fr', {})
        vals = [fr.get(k, 0.) * 100. for k in fr_keys]  # → % spikes/step
        ax2.bar(x + i * w, vals, w,
                label=f"{nt} {NEURON_CONFIGS[nt]['name']}",
                color=NEURON_CONFIGS[nt]['color'], alpha=0.88)
    ax2.set_title('Mean Firing Rate at Test Time (%)', fontweight='bold')
    ax2.set_xticks(x + w * 2)
    ax2.set_xticklabels(fr_lbls)
    ax2.set_ylabel('Firing Rate (% spikes per step)', fontsize=10)
    ax2.legend(fontsize=8, loc='upper right'); ax2.grid(True, alpha=0.25, axis='y')

    plt.tight_layout()
    plt.savefig('03_performance_and_firing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅  Saved: 03_performance_and_firing.png")

In [ ]:
def plot_spike_patterns_and_analysis(all_hist: dict):
    """
    3-row × 5-col figure:
      Row 1 : simulated voltage trace per neuron type
      Row 2 : training loss gradient (dLoss/d_epoch) curve
      Row 3 : accuracy improvement rate (dAcc/d_epoch)
    + ranking summary table panel.
    """
    n_types  = len(NEURON_KEYS)
    fig      = plt.figure(figsize=(20, 13))
    gs_outer = gridspec.GridSpec(3, n_types + 1, figure=fig,
                                  hspace=0.55, wspace=0.32,
                                  width_ratios=[1]*n_types + [1.4])
    fig.suptitle(
        'Brain Region Analysis + Loss/Accuracy Gradient Plots\n'
        'Hybrid CNN-SNN with Izhikevich Neurons — PlantVillage',
        fontsize=14, fontweight='bold'
    )
    ep = list(range(1, EPOCHS + 1))

    for col, nt in enumerate(NEURON_KEYS):
        cfg = NEURON_CONFIGS[nt]
        c   = cfg['color']
        h   = all_hist.get(nt, {})

        # ── Row 0 : Izhikevich voltage trace ──────────────────────────
        ax0 = fig.add_subplot(gs_outer[0, col])
        vs, ts = simulate_izhikevich(nt)
        ax0.plot(vs, color=c, lw=1.4, label='v(t)')
        for t in ts:
            ax0.axvline(t, color=c, alpha=0.25, lw=0.8)
        ax0.axvline(50, color='gray', lw=1, ls='--', alpha=0.5)
        ax0.text(55, 28, 'I_on', fontsize=7, color='gray')
        ax0.set_title(f"[{nt}]\n{cfg['name']}", fontsize=9,
                       fontweight='bold', color=c)
        ax0.set_ylim(-85, 40)
        ax0.set_xlabel('ms', fontsize=7); ax0.set_ylabel('mV', fontsize=7)
        ax0.tick_params(labelsize=6)
        ax0.grid(True, alpha=0.15)
        n_spk = len(ts)
        ax0.text(0.98, 0.05, f"ISI:{n_spk} spk", transform=ax0.transAxes,
                  fontsize=6, ha='right', color=c, fontweight='bold')

        if not h:
            continue

        # ── Row 1 : Loss gradient (Δloss per epoch) ───────────────────
        ax1 = fig.add_subplot(gs_outer[1, col])
        vl   = np.array(h['val_loss'])
        dloss = np.gradient(vl)
        ax1.bar(ep, dloss, color=c, alpha=0.7, width=0.8)
        ax1.axhline(0, color='black', lw=0.8)
        ax1.set_title('ΔVal Loss / Epoch', fontsize=8, fontweight='bold')
        ax1.set_xlabel('Epoch', fontsize=7); ax1.set_ylabel('Gradient', fontsize=7)
        ax1.tick_params(labelsize=6); ax1.grid(True, alpha=0.15, axis='y')

        # ── Row 2 : Accuracy improvement rate ─────────────────────────
        ax2 = fig.add_subplot(gs_outer[2, col])
        va   = np.array(h['val_acc'])
        dacc = np.gradient(va)
        ax2.bar(ep, dacc, color=c, alpha=0.7, width=0.8)
        ax2.axhline(0, color='black', lw=0.8)
        ax2.set_title('ΔVal Acc% / Epoch', fontsize=8, fontweight='bold')
        ax2.set_xlabel('Epoch', fontsize=7); ax2.set_ylabel('Δ%', fontsize=7)
        ax2.tick_params(labelsize=6); ax2.grid(True, alpha=0.15, axis='y')

    # ── Summary table panel ───────────────────────────────────────────
    ax_tbl = fig.add_subplot(gs_outer[:, -1])
    ax_tbl.axis('off')

    ranked = sorted(
        NEURON_KEYS,
        key=lambda nt: all_hist.get(nt, {}).get('test_acc', 0.),
        reverse=True
    )
    rank_icons = ['🥇', '🥈', '🥉', '4th', '5th']
    rows_data  = []
    for rk, nt in enumerate(ranked):
        h = all_hist.get(nt, {})
        rows_data.append([
            rank_icons[rk],
            nt,
            NEURON_CONFIGS[nt]['name'],
            f"{h.get('test_acc', 0.):.1f}%",
            f"{h.get('best_val_acc', 0.)*100:.1f}%",
        ])

    tbl = ax_tbl.table(
        cellText=rows_data,
        colLabels=['Rank', 'ID', 'Neuron Type', 'Test', 'Best Val'],
        loc='upper center', cellLoc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1.1, 1.8)

    # Colour rows by neuron type
    for rk, nt in enumerate(ranked):
        c = NEURON_CONFIGS[nt]['color']
        for j in range(5):
            cell = tbl[(rk+1, j)]
            cell.set_facecolor(c + '30')   # light tint

    ax_tbl.set_title('Performance Ranking\n& Firing Pattern Summary',
                      fontsize=10, fontweight='bold', pad=10)

    plt.savefig('04_spike_patterns_gradient_analysis.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅  Saved: 04_spike_patterns_gradient_analysis.png")

In [ ]:
# Dashboard

def plot_interactive_dashboard(all_hist: dict):
    """
    Full interactive Plotly dashboard:
      ① Val Loss curves (all 5 models)
      ② Val Accuracy curves
      ③ Train-Val gap (overfitting monitor)
      ④ Test accuracy bar chart
      ⑤ Firing-rate heatmap over epochs
    """
    nt_list = list(all_hist.keys())
    ep      = list(range(1, EPOCHS + 1))

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            '① Validation Loss', '② Validation Accuracy',
            '③ Train–Val Accuracy Gap (Overfitting)', '④ Final Test Accuracy',
            '⑤ Hidden-Layer Firing Rate over Epochs', '⑥ Integration Firing Rate',
        ],
        vertical_spacing=0.11, horizontal_spacing=0.08
    )

    for nt in nt_list:
        h   = all_hist[nt]
        c   = NEURON_CONFIGS[nt]['color']
        nm  = NEURON_CONFIGS[nt]['name']
        lbl = f"{nt} — {nm}"

        hover_ep = [f"<b>{lbl}</b><br>Epoch: {e}" for e in ep]

        # ① Val Loss
        fig.add_trace(go.Scatter(
            x=ep, y=h['val_loss'], name=lbl,
            line=dict(color=c, width=2.5), legendgroup=nt, showlegend=True,
            hovertemplate='%{text}<br>Val Loss: %{y:.4f}<extra></extra>',
            text=hover_ep,
        ), row=1, col=1)

        # ② Val Acc
        fig.add_trace(go.Scatter(
            x=ep, y=h['val_acc'], name=lbl,
            line=dict(color=c, width=2.5), legendgroup=nt, showlegend=False,
            hovertemplate='%{text}<br>Val Acc: %{y:.2f}%<extra></extra>',
            text=hover_ep,
        ), row=1, col=2)

        # ③ Train-Val gap
        gap = [tr - vl for tr, vl in zip(h['train_acc'], h['val_acc'])]
        fig.add_trace(go.Scatter(
            x=ep, y=gap, name=lbl,
            line=dict(color=c, width=2, dash='dot'), legendgroup=nt, showlegend=False,
            hovertemplate='%{text}<br>Gap: %{y:.2f}%<extra></extra>',
            text=hover_ep, fill='tozeroy',
            fillcolor=f"rgba{tuple(list(mcolors.to_rgb(c)) + [0.07])}",
        ), row=2, col=1)

        # ⑤⑥ Firing rates over epochs
        hid_fr  = [fr.get('hidden_fr', 0.) * 100.
                   for fr in h['firing_rates']]
        intg_fr = [fr.get('integrate_fr', 0.) * 100.
                   for fr in h['firing_rates']]
        fig.add_trace(go.Scatter(
            x=ep, y=hid_fr, name=lbl,
            line=dict(color=c, width=2), legendgroup=nt, showlegend=False,
            hovertemplate='%{text}<br>FR_hidden: %{y:.2f}%<extra></extra>',
            text=hover_ep,
        ), row=3, col=1)
        fig.add_trace(go.Scatter(
            x=ep, y=intg_fr, name=lbl,
            line=dict(color=c, width=2), legendgroup=nt, showlegend=False,
            hovertemplate='%{text}<br>FR_integ: %{y:.2f}%<extra></extra>',
            text=hover_ep,
        ), row=3, col=2)

    # ④ Test accuracy bar
    fig.add_trace(go.Bar(
        x=[f"<b>{nt}</b><br><sub>{NEURON_CONFIGS[nt]['name']}</sub>" for nt in nt_list],
        y=[all_hist[nt]['test_acc'] for nt in nt_list],
        marker_color=[NEURON_CONFIGS[nt]['color'] for nt in nt_list],
        text=[f"{all_hist[nt]['test_acc']:.2f}%" for nt in nt_list],
        textposition='auto', showlegend=False,
        hovertemplate="<b>%{x}</b><br>Test Accuracy: %{y:.2f}%<extra></extra>",
    ), row=2, col=2)

    fig.update_layout(
        title=dict(
            text='<b>🌿 Hybrid CNN-SNN Plant Disease Detection</b><br>'
                 '<sup>5 Izhikevich Neuron Types · PlantVillage Dataset · '
                 'Surrogate-Gradient + STDP Learning</sup>',
            font=dict(size=17), x=0.5, xanchor='center'
        ),
        height=950, width=1280,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.01,
                    xanchor='right', x=1, font=dict(size=10)),
        hoverlabel=dict(bgcolor='white', font_size=12),
    )

    # Axis labels
    axis_map = {
        (1,1): ('Epoch','Val Loss'),         (1,2): ('Epoch','Val Acc (%)'),
        (2,1): ('Epoch','Gap (%)'),          (2,2): ('Neuron Type','Test Acc (%)'),
        (3,1): ('Epoch','FR Hidden (%)'),    (3,2): ('Epoch','FR Integration (%)'),
    }
    for (r, c_), (xl, yl) in axis_map.items():
        fig.update_xaxes(title_text=xl, row=r, col=c_)
        fig.update_yaxes(title_text=yl, row=r, col=c_)
    fig.update_yaxes(range=[0, 105], row=2, col=2)

    fig.write_html('05_interactive_dashboard.html')
    fig.show()
    print("✅  Saved: 05_interactive_dashboard.html")



BRAIN REGION ANALYSIS REPORT

In [ ]:
def print_brain_analysis(all_hist: dict):
    """
    Console report: neuroscience context + image-classification suitability
    + which neuron type performs best and why.
    """
    best_nt  = max(all_hist, key=lambda nt: all_hist[nt]['test_acc'])
    worst_nt = min(all_hist, key=lambda nt: all_hist[nt]['test_acc'])

    sep = "═" * 75

    print(f"\n{sep}")
    print("  BRAIN REGION ANALYSIS & IMAGE-PROCESSING SUITABILITY")
    print(f"  Hybrid CNN-SNN · PlantVillage Disease Classification")
    print(sep)

    for nt in NEURON_KEYS:
        cfg  = NEURON_CONFIGS[nt]
        h    = all_hist.get(nt, {})
        icon = "⭐" if nt == best_nt else ("❌" if nt == worst_nt else "  ")
        print(f"\n  {icon} [{nt}]  {cfg['name']}")
        print(f"     Params      : a={cfg['a']}, b={cfg['b']}, "
              f"c={cfg['c']}, d={cfg['d']}")
        print(f"     Brain Region: {cfg['brain_region']}")
        print(f"     Test Acc    : {h.get('test_acc', 0.):.2f}%  "
              f"(best val: {h.get('best_val_acc',0.)*100:.2f}%)")
        print(f"     Neuroscience:")
        for ln in cfg['neuroscience'].split('. '):
            if ln.strip():
                print(f"       → {ln.strip()}.")
        print(f"     Image Suitability:")
        for ln in cfg['image_suitability'].split('. '):
            if ln.strip():
                print(f"       • {ln.strip()}.")

    print(f"\n{sep}")
    print(f"  🏆  BEST MODEL  : [{best_nt}] {NEURON_CONFIGS[best_nt]['name']}")
    print(f"      Test Acc    : {all_hist[best_nt]['test_acc']:.2f}%")
    print()
    print("  WHY CHATTERING (CH) NEURONS EXCEL AT IMAGE CLASSIFICATION:")
    print("  ─────────────────────────────────────────────────────────")
    print("  1. BURST ENCODING: Each 300–700 Hz burst within the SNN timestep")
    print("     window T encodes a 'feature packet' — a compact high-bandwidth")
    print("     representation of local image statistics (texture, curvature).")
    print()
    print("  2. TEMPORAL SEGMENTATION: The burst-then-silence rhythm naturally")
    print("     separates foreground (disease lesion) features from background")
    print("     (healthy leaf tissue), acting as a temporal attention mechanism.")
    print()
    print("  3. BIOLOGICAL ALIGNMENT: CH neurons were first found in cat V1/V2")
    print("     (Gray & McCormick 1996) precisely in the region that performs")
    print("     figure-ground segregation and perceptual binding — the exact")
    print("     computation needed for isolating disease patches on leaves.")
    print()
    print("  4. PARAMETER c=−50: The depolarised reset voltage allows rapid")
    print("     re-activation, meaning the network can encode rich multi-scale")
    print("     feature sequences within the short T timestep window.")
    print()
    print("  5. GAMMA SYNCHRONY: CH neurons drive gamma-band (30–80 Hz)")
    print("     oscillations that bind spatially distributed features into a")
    print("     coherent percept — analogous to how the CNN+SNN pipeline")
    print("     integrates low-level textures into high-level disease labels.")
    print(f"\n{sep}")

    # Ranked table
    ranked = sorted(NEURON_KEYS,
                    key=lambda nt: all_hist.get(nt,{}).get('test_acc',0.),
                    reverse=True)
    print("\n  FINAL RANKING:")
    for i, nt in enumerate(ranked):
        acc = all_hist.get(nt,{}).get('test_acc',0.)
        region = NEURON_CONFIGS[nt]['brain_region'].split('(')[0].strip()
        print(f"    {i+1}. [{nt}] {NEURON_CONFIGS[nt]['name']:24s} "
              f"| {acc:.2f}% | {region}")
    print(sep)

In [ ]:
def main():
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  🌿 Plant Disease Detection · Hybrid CNN-SNN               ║")
    print("║  5 Izhikevich Neuron Types · STDP + Surrogate-Gradient     ║")
    print("╚══════════════════════════════════════════════════════════════╝\n")

    # ── 1. Dataset ────────────────────────────────────────────────────
    try:
        loaders, num_classes, splits = load_plantvillage(DATA_ROOT, SUBSET_FRAC)
    except FileNotFoundError as e:
        print(e)
        print("\nUpdate DATA_ROOT at the top of this script after downloading.")
        return

    # ── 2. Model parameter overview ───────────────────────────────────
    print("\nModel parameter counts:")
    for nt in NEURON_KEYS:
        make_model(nt, num_classes)

    # ── 3. Train all 5 models ─────────────────────────────────────────
    all_hist = train_all(loaders, num_classes, epochs=EPOCHS)

    # ── 4. Visualisations ─────────────────────────────────────────────
    print("\n📊  Generating plots …")
    plot_per_model_curves(all_hist)
    plot_comparison_overlay(all_hist)
    plot_bar_and_firing(all_hist)
    plot_spike_patterns_and_analysis(all_hist)
    plot_interactive_dashboard(all_hist)

    # ── 5. Analysis ───────────────────────────────────────────────────
    print_brain_analysis(all_hist)

    # ── 6. Save JSON history ──────────────────────────────────────────
    def _json_safe(v):
        if isinstance(v, (np.float32, np.float64, np.int32, np.int64)):
            return float(v)
        if isinstance(v, dict):
            return {k: _json_safe(vv) for k, vv in v.items()}
        if isinstance(v, list):
            return [_json_safe(vv) for vv in v]
        return v

    out = {}
    for nt, h in all_hist.items():
        # exclude large per-epoch firing_rates list from JSON for brevity
        out[nt] = {k: _json_safe(v) for k, v in h.items()
                   if k != 'firing_rates'}
    with open('all_histories.json', 'w') as f:
        json.dump(out, f, indent=2)
    print("\n✅  All done!  Output files:")
    print("    01_per_model_curves.png")
    print("    02_comparison_overlay.png")
    print("    03_performance_and_firing.png")
    print("    04_spike_patterns_gradient_analysis.png")
    print("    05_interactive_dashboard.html")
    print("    all_histories.json")
    print("    ckpt_RS.pt  ckpt_FS.pt  ckpt_CH.pt  ckpt_IB.pt  ckpt_LTS.pt")


if __name__ == '__main__':
    main()
